# Day 3 — Gemma 4 E4B: C1 vs C2 QLoRA fine-tuning

**Research:** Learning to Recover: Teaching Small Coding Agents to Fix Their Own Mistakes Using Execution Feedback

**Purpose:** Train **C1 (success-only)** and **C2 (recovery-aware)** adapters, starting from **the same original Gemma 4 E4B-it weights**. Keep the number of supervised examples and optimizer steps the same and closely match the supervised *target token* counts. Export two LoRA adapters for Day 4 evaluation.

**Input:** Your saved Day 2 `gemma_day2_backup.zip` attached using Kaggle **Add Input**, or the attached Day 2 notebook's saved output files. **Do not rerun Days 1 or 2.**

**GPU:** Kaggle **GPU T4 ×2** (we deliberately use **GPU 0 only**). Internet on for first model download; Hugging Face license access if required. No tokens/secrets should be pasted into notebook cells.

**Safety:** This notebook reads JSONL and trains; it does *not* execute generated Python. The Day 2 verification was only on the original MBPP visible tests; no MBPP+/HumanEval+ scores are reported here.

**Practical sequencing:** Run A–E (data/tokenization), then F (C1), then G (C2), H–I (validate/backup). **If C1 fails with CUDA OOM, do not start C2**; adjust `MAX_SEQ_LEN` downward, restart the session and rerun from A. Training progress appears through the built-in `tqdm` progress bars.
**T4 OOM revision:** The frozen 4-bit base is no longer prepared through PEFT's full fp32 upcast. The training settings and C1/C2 examples are otherwise unchanged.


## A — Dependencies and GPU check

Install packages once per new Kaggle session. Kaggle may print pip dependency warnings. After installing, verify imports; restart the kernel **only** if imports behave incorrectly. Run this cell before importing Transformers or PEFT.

In [ ]:
%pip install -q -U "transformers>=5.18.0" "peft>=0.19.0" accelerate datasets bitsandbytes tqdm safetensors

In [ ]:
# Restrict the notebook to one T4 before importing torch/transformers.
# Trainer otherwise may DataParallel both GPUs, incompatible with device_map.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import torch, transformers, peft, datasets
print('Torch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('PEFT:', peft.__version__)
print('CUDA:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print('GPU', i, torch.cuda.get_device_name(i),
          f'{torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GiB')
assert torch.cuda.is_available(), 'Please select Kaggle GPU accelerator (T4 x2).'
assert torch.cuda.device_count() == 1, 'Use fresh kernel with only GPU 0 visible (set CUDA_VISIBLE_DEVICES before torch import).'

## B — Create helper module

This notebook contains the actual Day 3 code. It does **not** depend on an external GitHub repository or on editing your original Day 1/Day 2 notebooks.

The helper provides input discovery, split verification, exact assistant-completion masking, matched-token sampling, model loading, LoRA fine-tuning, and resumable Trainer checkpoints.

In [ ]:
%%writefile /kaggle/working/day3_train.py
"""Day 3 matched-budget QLoRA for Gemma 4 E4B; uses Day 2 prompt/completion JSONL."""
from collections import Counter
from pathlib import Path
import hashlib, json, random, shutil


def read_jsonl(path):
    return [json.loads(x) for x in Path(path).read_text(encoding='utf-8').splitlines() if x.strip()]


def write_json(path, value):
    p=Path(path);p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(value, indent=2, ensure_ascii=False),encoding='utf-8')


def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1<<20),b''): h.update(block)
    return h.hexdigest()


def discover_day2_input(root='/kaggle/input', output='/kaggle/working/day3_input'):
    """Discover Day 2 files via Kaggle Add Input or its saved backup ZIP."""
    from zipfile import ZipFile
    root,output=Path(root),Path(output)
    req={'success_only.jsonl','recovery.jsonl','dataset_summary.json'}
    if output.is_dir() and req <= {p.name for p in output.iterdir() if p.is_file()}:
        return output
    if root.exists():
        for p in sorted(root.rglob('dataset_summary.json')):
            if p.is_file() and req <= {q.name for q in p.parent.iterdir() if q.is_file()}:
                output.mkdir(parents=True,exist_ok=True)
                for name in ['success_only.jsonl','recovery.jsonl','recovery_self.jsonl',
                             'recovery_reference.jsonl','c2_mixed.jsonl',
                             'dataset_summary.json','data_collection_config.json']:
                    src=p.parent/name
                    if src.is_file(): shutil.copy2(src,output/name)
                ms=list(root.rglob('mbpp_split_manifest.json'))
                if ms: shutil.copy2(ms[0],output/'mbpp_split_manifest.json')
                write_json(output/'input_location.json',{'source':str(p.parent),'mode':'loose_files'})
                return output
    zips=sorted(root.rglob('gemma_day2_backup*.zip')) if root.exists() else []
    if not zips: raise FileNotFoundError('Attach the Day 2 saved outputs OR the Day 2 backup ZIP via Kaggle Add Input. Searched '+str(root))
    zip_path=zips[0]; output.mkdir(parents=True,exist_ok=True)
    with ZipFile(zip_path) as z:
        assert z.testzip() is None,'Corrupt Day 2 archive'
        found=set()
        for member in z.infolist():
            parts=Path(member.filename).parts
            if '__MACOSX' in parts or member.is_dir(): continue
            basename=parts[-1]
            if 'day2_data' in parts and basename in {'success_only.jsonl','recovery.jsonl',
                'recovery_self.jsonl','recovery_reference.jsonl','c2_mixed.jsonl',
                'dataset_summary.json','data_collection_config.json'}:
                (output/basename).write_bytes(z.read(member));found.add(basename)
            if basename=='mbpp_split_manifest.json':
                (output/basename).write_bytes(z.read(member))
        if not req <= found: raise ValueError(f'Day 2 ZIP missing files: {req-found}')
    write_json(output/'input_location.json',{'source':str(zip_path),'mode':'zip','sha256':sha256(zip_path)})
    return output


def validate_day2(data_dir):
    d=Path(data_dir)
    success=read_jsonl(d/'success_only.jsonl');recovery=read_jsonl(d/'recovery.jsonl')
    summary=json.loads((d/'dataset_summary.json').read_text())
    assert success and recovery,'Empty training pool'
    assert len(success)==summary['c1_success_examples']
    assert len(recovery)==summary['c2_recovery_combined_examples']
    for row in success+recovery:
        assert row.get('prompt') and row.get('completion') and row.get('target_code')
        assert row['completion'][-1]['role']=='assistant'
        assert row['completion'][-1]['content']==row['target_code']
        assert row.get('verified_on')=='original MBPP visible tests only'
    assert not summary.get('evaluation_overlap_ids'), 'Day 2 reported benchmark overlap'
    mpath=d/'mbpp_split_manifest.json'
    if mpath.exists():
        manifest=json.loads(mpath.read_text())
        approved=set(manifest['mbppplus_decontaminated']['train']['task_ids'])
        held_out=set(manifest['evalplus_mbppplus_ids'])
        actual={int(r['task_id']) for r in success+recovery}
        assert actual <= approved, f'Not in eligible training split: {sorted(actual-approved)}'
        assert actual.isdisjoint(held_out), f'Benchmark contamination: {sorted(actual & held_out)}'
    return {'success':success,'recovery':recovery,'summary':summary}


def _ids(processor,messages,add_generation_prompt):
    ans=processor.apply_chat_template(messages,tokenize=True,return_dict=True,
        return_tensors='pt',add_generation_prompt=add_generation_prompt,enable_thinking=False)
    x=ans['input_ids']
    return x[0].tolist() if getattr(x,'ndim',None)==2 else list(x)


def tokenize_example(row,processor,max_seq_len):
    """Mask all prompt tokens (including earlier assistant mistakes)."""
    prefix=_ids(processor,row['prompt'],True)
    full=_ids(processor,row['prompt']+row['completion'],False)
    if full[:len(prefix)]!=prefix:
        raise ValueError('Chat template prefix mismatch. STOP: inspect Gemma tokenizer formatting; do not train on wrong labels.')
    target=full[len(prefix):]
    assert target,'No supervised tokens'
    if len(full)>max_seq_len:return None
    labels=[-100]*len(prefix)+target
    assert len(labels)==len(full)
    return {'task_id':int(row['task_id']),'sample_index':row.get('sample_index'),
        'example_type':row['example_type'],'source':row['source'],
        'input_ids':full,'attention_mask':[1]*len(full),'labels':labels,
        'target_tokens':len(target),'prompt_tokens':len(prefix),'sequence_tokens':len(full)}


def tokenize_pools(data,processor,max_seq_len):
    pools={'success':[],'recovery':[]};dropped={'success':[],'recovery':[]}
    from tqdm.auto import tqdm
    for k in pools:
        for r in tqdm(data[k],desc=f'Tokenize {k}',unit='example'):
            t=tokenize_example(r,processor,max_seq_len)
            if t is None:dropped[k].append({'task_id':r['task_id'],'source':r['source']})
            else:pools[k].append(t)
        print(k,len(pools[k]),'fit;',len(dropped[k]),'too long')
    if not pools['success'] or not pools['recovery']:
        raise ValueError('Insufficient fitting examples; inspect MAX_SEQ_LEN')
    return {'pools':pools,'dropped':dropped}


def matched_plans(tokenized,seed=42,tries=6000,tolerance=0.04):
    """Same sample count / optimizer steps, matched supervised *target* tokens.

    C1 = all available success rows. C2 = every recovery row plus a
    subset of success rows selected to match C1's target-token count.
    """
    success=tokenized['pools']['success'];recovery=tokenized['pools']['recovery']
    if len(recovery)>len(success):raise ValueError('Recovery pool exceeds success pool')
    n=len(success);k=n-len(recovery)
    target=sum(r['target_tokens'] for r in success)
    r_tokens=sum(r['target_tokens'] for r in recovery)
    rng=random.Random(seed);ids=list(range(n));best=None
    for _ in range(max(1,tries)):
        chosen=rng.sample(ids,k)
        observed=r_tokens+sum(success[i]['target_tokens'] for i in chosen)
        delta=abs(observed-target)
        if best is None or delta<best[0]:best=(delta,tuple(sorted(chosen)),observed)
        if delta==0:break
    c1=list(success);c2=list(recovery)+[success[i] for i in best[1]]
    rng1=random.Random(seed+101);rng2=random.Random(seed+202)
    rng1.shuffle(c1);rng2.shuffle(c2)
    delta=abs(best[2]-target)/target
    print('C1:',len(c1),'rows,',target,'target tokens')
    print('C2:',len(c2),'rows,',best[2],'target tokens')
    print('Token difference:',f'{delta:.2%}','| C2 sources:',dict(Counter(r['source'] for r in c2)))
    if delta>tolerance:
        raise ValueError(f'C1 vs C2 target tokens differ by {delta:.2%}, exceeding {tolerance:.0%}. ' 
                         'Change MAX_SEQ_LEN or explicit matching strategy before training.')
    manifest={
        'seed':seed,'c1_count':len(c1),'c2_count':len(c2),
        'c1_target_tokens':target,'c2_target_tokens':best[2],
        'relative_token_difference':delta,
        'c1_sources':dict(Counter(r['source'] for r in c1)),
        'c2_sources':dict(Counter(r['source'] for r in c2)),
        'c1_task_ids':sorted(set(r['task_id'] for r in c1)),
        'c2_task_ids':sorted(set(r['task_id'] for r in c2)),
        'c1_rows':[{'task_id':r['task_id'],'sample_index':r['sample_index'],
                    'source':r['source'],'target_tokens':r['target_tokens']} for r in c1],
        'c2_rows':[{'task_id':r['task_id'],'sample_index':r['sample_index'],
                    'source':r['source'],'target_tokens':r['target_tokens']} for r in c2],
        'sequence_exclusions':tokenized['dropped'],
        'notes':['C1 contains all fitting success-only rows.',
                 'C2 contains all fitting recovery rows and selected success rows.',
                 'Same training examples count and gradient updates; target tokens closely matched.',
                 'Different prompt lengths are not controlled; report prompt/computation costs separately.',
                 'Recovery corrections include separately labeled self and reference sources.',
                 'All training examples were verified on original MBPP visible tests only.']}
    return {'c1':c1,'c2':c2,'manifest':manifest}


class CompletionCollator:
    def __init__(self,pad_token_id):self.pad=pad_token_id
    def __call__(self,features):
        import torch
        n=max(len(f['input_ids']) for f in features)
        ids=[r['input_ids']+[self.pad]*(n-len(r['input_ids'])) for r in features]
        am=[r['attention_mask']+[0]*(n-len(r['attention_mask'])) for r in features]
        labels=[r['labels']+[-100]*(n-len(r['labels'])) for r in features]
        return {'input_ids':torch.tensor(ids,dtype=torch.long),
                'attention_mask':torch.tensor(am,dtype=torch.long),
                'labels':torch.tensor(labels,dtype=torch.long)}


def build_dataset(plan):
    from datasets import Dataset
    cols=['input_ids','attention_mask','labels']
    return Dataset.from_dict({k:[r[k] for r in plan] for k in cols})


def load_fresh_model(model_id,lora_rank=8):
    import torch
    from transformers import AutoModelForMultimodalLM,BitsAndBytesConfig
    from peft import LoraConfig,get_peft_model
    quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
    base=AutoModelForMultimodalLM.from_pretrained(
        model_id,quantization_config=quant,device_map={'':0},dtype=torch.float16)
    base.config.use_cache=False
    # T4 VRAM fix: avoid PEFT casting huge frozen Gemma 4 embedding tables to FP32.
    # Retain 4-bit / fp16 weights; freeze manually and enable checkpointing.
    for param in base.parameters():
        param.requires_grad_(False)
    base.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={'use_reentrant': False}
    )
    if hasattr(base, 'enable_input_require_grads'):
        base.enable_input_require_grads()
    print('Base frozen; gradient checkpointing enabled (no FP32 embedding upcast).')
    print('GPU allocated:',
          round(torch.cuda.memory_allocated(0) / 1024**3, 2), 'GiB')
    # Explicit regex limits LoRA to *language* attention, not vision/audio layers.
    lora=LoraConfig(task_type='CAUSAL_LM',r=lora_rank,lora_alpha=2*lora_rank,
        lora_dropout=0.05,bias='none',
        target_modules=r'.*language_model.*\.(q_proj|v_proj)')
    model=get_peft_model(base,lora)
    model.config.use_cache=False
    # Some Gemma4 Transformers releases mishandle loss normalization with gradient accumulation.
    model.accepts_loss_kwargs=False
    model.print_trainable_parameters()
    n=sum(p.numel() for p in model.parameters() if p.requires_grad)
    if n<1000:raise RuntimeError('Unexpectedly no trainable language-model LoRA weights')
    return model


def train_condition(condition,plan,model_id,output_root,seed=42,max_seq_len=1024,
                    lr=1e-4,epochs=1,grad_accum=4,lora_rank=8,
                    save_steps=5,resume=False):
    """Train C1/C2 independently; use built-in Trainer tqdm and checkpoints."""
    import gc,torch
    from transformers import Trainer,TrainingArguments,AutoProcessor,set_seed
    from transformers.trainer_utils import get_last_checkpoint
    root=Path(output_root);root.mkdir(parents=True,exist_ok=True)
    out=root/f'train_{condition}';adapter=root/f'adapter_{condition}'
    if (adapter/'adapter_config.json').exists():
        raise RuntimeError(f'{adapter} already exists. Do not overwrite a completed adapter.')
    set_seed(seed);torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(0)
    model=load_fresh_model(model_id,lora_rank=lora_rank)
    model.train()
    processor=AutoProcessor.from_pretrained(model_id)
    pad=processor.tokenizer.pad_token_id
    if pad is None:pad=processor.tokenizer.eos_token_id
    assert pad is not None
    ds=build_dataset(plan)
    args=TrainingArguments(
        output_dir=str(out),
        per_device_train_batch_size=1,gradient_accumulation_steps=grad_accum,
        num_train_epochs=epochs,learning_rate=lr,lr_scheduler_type='constant',
        optim='adamw_torch',weight_decay=0.0,
        fp16=True,bf16=False, # T4 GPUs do not support bf16 natively
        logging_strategy='steps',logging_steps=1,
        save_strategy='steps',save_steps=save_steps,save_total_limit=2,
        remove_unused_columns=False,dataloader_num_workers=0,
        disable_tqdm=False,report_to='none',seed=seed,data_seed=seed,
        max_grad_norm=0.3,gradient_checkpointing=False, # handled above
    )
    trainer=Trainer(model=model,args=args,train_dataset=ds,
                    data_collator=CompletionCollator(pad))
    checkpoint=get_last_checkpoint(str(out)) if out.exists() and resume else None
    if checkpoint:print('Resuming from',checkpoint,flush=True)
    result=trainer.train(resume_from_checkpoint=checkpoint)
    trainer.save_model(str(adapter))
    metrics=dict(result.metrics)
    metrics.update({'condition':condition,'base_model':model_id,
        'num_examples':len(plan),
        'supervised_target_tokens_per_epoch':sum(r['target_tokens'] for r in plan),
        'max_seq_len':max_seq_len,'learning_rate':lr,'epochs':epochs,
        'gradient_accumulation_steps':grad_accum,'lora_rank':lora_rank,
        'seed':seed,'adapter_path':str(adapter),
        'gpu0_peak_allocated_gib':round(torch.cuda.max_memory_allocated(0)/1024**3,3)})
    write_json(root/f'{condition}_metrics.json',metrics)
    write_json(root/f'{condition}_log_history.json',trainer.state.log_history)
    print('FINISHED',condition,'training_loss:',metrics.get('train_loss'),flush=True)
    del trainer,model,processor,ds
    gc.collect();torch.cuda.empty_cache()
    return metrics

## C — Attach and verify your Day 2 data

In **Add Input**, attach either your Day 2 notebook **Outputs**, or a private Kaggle Dataset containing the downloaded `gemma_day2_backup.zip`.

The next cell searches `/kaggle/input` and extracts only the required Day 2 data files into `/kaggle/working/day3_input`. The original input remains unchanged. This ZIP may contain a containing folder and macOS metadata; both are handled.

In [ ]:
from pathlib import Path
from day3_train import discover_day2_input, validate_day2, sha256, write_json

DATA_DIR = discover_day2_input()
day2 = validate_day2(DATA_DIR)
print('Input directory:', DATA_DIR)
print('Success-only examples:', len(day2['success']))
print('Recovery examples:', len(day2['recovery']))
print('Recovery sources:')
from collections import Counter
print(dict(Counter(row['source'] for row in day2['recovery'])))
print('No MBPP+ task-ID overlap in training input: VERIFIED')
input_hashes={f.name:sha256(f) for f in DATA_DIR.iterdir() if f.is_file()}
write_json(Path('/kaggle/working/day3_input_checksums.json'), input_hashes)
print('Day 2 input hashes saved for reproducibility')

## D — Configuration and *completion-only* tokenization

This uses the **same Gemma 4 E4B-it checkpoint** and instruction-tuned chat template as Days 1 and 2. The tokenizer sees the problem, old failed assistant response, and error feedback; labels for all those tokens are `-100`. **Only the new successful assistant completion contributes to loss.**

**Important:** If the actual chat-template prefix doesn't match the full response, the code stops instead of silently training with incorrect masks. If long samples are excluded, the IDs and counts are recorded in the training manifest.

In [ ]:
from transformers import AutoProcessor
from day3_train import tokenize_pools, matched_plans, write_json

MODEL_ID = 'google/gemma-4-E4B-it'
SEED = 42
MAX_SEQ_LEN = 1024   # On a T4: if CUDA OOM, try 768 (fewer long examples may fit)
LORA_RANK = 8
LEARNING_RATE = 1e-4
EPOCHS = 1
GRAD_ACCUM = 4
SAVE_STEPS = 5
OUT = Path('/kaggle/working/day3_outputs')
OUT.mkdir(parents=True, exist_ok=True)

processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenized = tokenize_pools(day2, processor, MAX_SEQ_LEN)
plans = matched_plans(tokenized, seed=SEED, tries=6000, tolerance=0.04)
write_json(OUT / 'training_plan.json', plans['manifest'])
print('Training plan saved:', OUT / 'training_plan.json')

## E — Check loss masking and GPU budget **before training**

These sanity checks ensure that each example has nonzero target tokens, all earlier tokens are masked, and both plans have equal sample counts and nearly equal target-token counts.  Training cost also depends on prompt lengths, which are **not exactly matched**; we'll report that limitation.

In [ ]:
from collections import Counter
from day3_train import CompletionCollator

for condition in ('c1', 'c2'):
    rows = plans[condition]
    assert rows, f'{condition}: empty dataset'
    for item in rows:
        p = item['prompt_tokens']
        assert len(item['input_ids']) <= MAX_SEQ_LEN
        assert all(x == -100 for x in item['labels'][:p])
        assert all(x >= 0 for x in item['labels'][p:])
        assert sum(x != -100 for x in item['labels']) == item['target_tokens']
    print(condition, 'examples:', len(rows),
          'supervised tokens:', sum(x['target_tokens'] for x in rows),
          'total prompt tokens:', sum(x['prompt_tokens'] for x in rows),
          'source mix:', dict(Counter(x['source'] for x in rows)))

assert len(plans['c1']) == len(plans['c2']), 'Sample counts must match'
assert plans['manifest']['relative_token_difference'] <= 0.04
num_recoveries = sum(r['example_type'] == 'recovery' for r in plans['c2'])
print('Recovery examples in C2:', num_recoveries)
assert num_recoveries >= 5, 'Too few recovery examples; consider longer MAX_SEQ_LEN if memory permits'

pad = processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id
smoke_batch = CompletionCollator(pad)([plans['c2'][0]])
print('Batch shapes:', {k: tuple(v.shape) for k,v in smoke_batch.items()})
print('Loss-bearing tokens:', int((smoke_batch['labels'] != -100).sum()))
print('All checks passed. Ready for C1 training.')
# Fail early on API changes before loading the 8B-parameter model.
import inspect
from transformers import TrainingArguments
required_args = {"output_dir", "save_strategy", "save_steps", "save_total_limit", "fp16", "gradient_accumulation_steps", "logging_strategy", "logging_steps"}
missing_args = required_args - set(inspect.signature(TrainingArguments).parameters)
assert not missing_args, f"Unsupported TrainingArguments parameters: {missing_args}"
print("TrainingArguments compatibility preflight passed.")


## F — Train C1: success-only Gemma adapter

**Important:** Fresh pretrained 4-bit weights are loaded for C1; LoRA is injected into **language-model q/v attention projections only**. Model-generated and reference success examples are both labeled in the training plan.

This will display a **`tqdm` training progress bar** with steps, ETA and loss logging. It saves periodic `checkpoint-*` folders, plus final `adapter_c1` and `c1_metrics.json`. GPU 0 is used deliberately; the second T4 is not automatically combined. Don't rerun this cell after it successfully completes.
**T4 OOM fix:** We skip `prepare_model_for_kbit_training()` because it upcasts Gemma 4 E4B embedding tables and can exhaust 14.56 GiB VRAM before training. We freeze base parameters and enable gradient checkpointing directly. If actual training-step OOM occurs, use `MAX_SEQ_LEN = 768`, rerun sections D–E and train again. Not GPU-validated yet.


In [ ]:
from day3_train import train_condition

c1_metrics = train_condition(
    condition='c1',
    plan=plans['c1'],
    model_id=MODEL_ID,
    output_root=OUT,
    seed=SEED,
    max_seq_len=MAX_SEQ_LEN,
    lr=LEARNING_RATE,
    epochs=EPOCHS,
    grad_accum=GRAD_ACCUM,
    lora_rank=LORA_RANK,
    save_steps=SAVE_STEPS,
    resume=True,  # resume only if a Trainer checkpoint already exists
)
print('C1 adapter saved:', OUT / 'adapter_c1')

## G — Train C2: recovery-aware Gemma adapter

**Start this only after C1 succeeded and its adapter was saved.** This reloads the original Gemma weights, not the C1 adapter. C2 uses the matched plan containing validated recovery examples and selected success-only examples.

The difference between the two training conditions is the **training mixture**. This does not constitute a controlled study of every possible mixture ratio, so we will report it accurately in the paper.

In [ ]:
assert (OUT / 'adapter_c1' / 'adapter_config.json').exists(), 'Finish C1 training first.'

c2_metrics = train_condition(
    condition='c2',
    plan=plans['c2'],
    model_id=MODEL_ID,
    output_root=OUT,
    seed=SEED,
    max_seq_len=MAX_SEQ_LEN,
    lr=LEARNING_RATE,
    epochs=EPOCHS,
    grad_accum=GRAD_ACCUM,
    lora_rank=LORA_RANK,
    save_steps=SAVE_STEPS,
    resume=True,
)
print('C2 adapter saved:', OUT / 'adapter_c2')

## H — Verify artifacts and training logs

Training loss shows that LoRA optimized its supervised objective; it does **not** prove that recovery on unseen MBPP+/HumanEval+ improved. That comparison comes on Day 4.

In [ ]:
import json
from day3_train import write_json

for condition in ('c1','c2'):
    folder=OUT/f'adapter_{condition}'
    assert (folder/'adapter_config.json').exists(), f'Missing adapter config for {condition}'
    assert any(folder.glob('*.safetensors')), f'Missing adapter weights for {condition}'
    stats=json.loads((OUT/f'{condition}_metrics.json').read_text())
    print(condition.upper(),
          'loss:', stats.get('train_loss'),
          '| examples:', stats['num_examples'],
          '| target tokens:',stats['supervised_target_tokens_per_epoch'],
          '| max GPU0 GiB:',stats['gpu0_peak_allocated_gib'])

summary = {
    'base_model':MODEL_ID,'seed':SEED,'max_seq_len':MAX_SEQ_LEN,
    'method':'QLoRA NF4, fp16 on T4, language_model q_proj/v_proj',
    'epochs':EPOCHS,'lora_rank':LORA_RANK,
    'learning_rate':LEARNING_RATE,'grad_accum':GRAD_ACCUM,
    'c1':json.loads((OUT/'c1_metrics.json').read_text()),
    'c2':json.loads((OUT/'c2_metrics.json').read_text()),
    'token_matched_training_plan':plans['manifest'],
    'note':'No held-out code correctness was measured on Day 3.',
}
write_json(OUT/'day3_summary.json',summary)
print('Day 3 artifact verification PASSED.')

## I — ZIP backup: adapters + training configuration, metrics and reproducibility logs

The ZIP contains **LoRA adapters only**, not an entire Gemma checkpoint. The original `google/gemma-4-E4B-it` model must be reloaded for Day 4. The trainer checkpoint optimizer states may be larger; the backup includes final adapters + metrics/manifest and helper code, but deliberately omits intermediate optimizer checkpoints. If you need an exact interrupted-run resume, also save `train_c1/checkpoint-*` or `train_c2/checkpoint-*` in a Kaggle version.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display
from pathlib import Path

BACKUP = Path('/kaggle/working/gemma_day3_backup.zip')
required=[OUT/'day3_summary.json',OUT/'training_plan.json',
          OUT/'c1_metrics.json',OUT/'c2_metrics.json',
          OUT/'c1_log_history.json',OUT/'c2_log_history.json',
          Path('/kaggle/working/day3_train.py'),
          Path('/kaggle/working/day3_input_checksums.json')]
for cond in ('c1','c2'):
    required += list((OUT/f'adapter_{cond}').rglob('*'))
required=[p for p in required if p.is_file()]
assert (OUT/'adapter_c1'/'adapter_config.json') in required
assert (OUT/'adapter_c2'/'adapter_config.json') in required
with ZipFile(BACKUP,'w',ZIP_DEFLATED) as z:
    for path in required:
        z.write(path, arcname=str(path.relative_to('/kaggle/working')))
with ZipFile(BACKUP,'r') as z:
    assert z.testzip() is None, 'Backup corruption detected!'
    print('Backup integrity verified. Files:',len(z.namelist()))
print('Backup size (MB):',round(BACKUP.stat().st_size/1024**2,2))
print('Backup created:',BACKUP)
display(FileLink(str(BACKUP)))

## After Day 3

1. **Download** `gemma_day3_backup.zip` before ending the session.
2. In Kaggle, **Save Version → Quick Save** (retain outputs if available) and verify the saved version.
3. Day 4: load the base Gemma 4 E4B-it checkpoint plus either `adapter_c1` or `adapter_c2`. Compare against C0 on the **same held-out tasks, same feedback and same attempt limits**.
4. Keep the Day 1 and Day 2 backups unchanged.

### Troubleshooting

- **Out of memory during training:** Do not continue to C2. Reduce `MAX_SEQ_LEN` to 768 (or 512 as last resort), *restart the runtime*, regenerate the plans and retrain both conditions consistently. Use one GPU, rank 8, microbatch 1, accumulation 4.
- **Prompt mask mismatch:** Stop. The Gemma 4 processor/chat-template version may have changed. Inspect the prefix and full encodings before training; do not set `labels=input_ids` as a workaround.
- **Resume after interrupted C1/C2:** Reattach a saved checkpoint directory containing optimizer/scheduler state, ensure the config is the same, and rerun that condition with `resume=True`. Merely restoring a final adapter does not reproduce an interrupted training run.
- **Saved version shows no files:** You still have the independently downloaded ZIP; validate ZIP contents before closing Kaggle.
- **Import fails after pip upgrade:** Restart the kernel once and rerun notebook from A, rather than repeatedly upgrading PyTorch.
- **Quantized LoRA failing on vision/audio modules:** The provided target regex selects language-model q/v only; do not replace it with an unscoped `all-linear` target.